In [1]:
import pandas as pd

# Read CSV
df = pd.read_csv("../data/raw/02_nav_history.csv")

# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Sort by fund and date
df = df.sort_values(by=["amfi_code", "date"])

# Check missing NAV values before filling
print("Missing NAV before fill:", df["nav"].isna().sum())

# Forward-fill NAV within each fund
df["nav"] = df.groupby("amfi_code")["nav"].ffill()

# Check missing NAV values after filling
print("Missing NAV after fill:", df["nav"].isna().sum())

# Remove duplicate rows
rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)

print("Rows before removing duplicates:", rows_before)
print("Rows after removing duplicates:", rows_after)

# Keep only valid NAV values
df = df[df["nav"] > 0]

# Verify no invalid NAV values remain
print("Invalid NAV rows:", (df["nav"] <= 0).sum())

# Final row count
print("Final row count:", len(df))

# Save cleaned dataset
df.to_csv("../data/processed/nav_history_clean.csv", index=False)

print("Cleaned file saved successfully!")

Missing NAV before fill: 0
Missing NAV after fill: 0
Rows before removing duplicates: 46000
Rows after removing duplicates: 46000
Invalid NAV rows: 0
Final row count: 46000
Cleaned file saved successfully!


In [25]:
import pandas as pd

# Read file
df_investor = pd.read_csv("../data/raw/08_investor_transactions.csv")

# Standardize transaction_type
transaction_map = {
    'sip': 'SIP',
    'lumpsum': 'Lumpsum',
    'redemption': 'Redemption'
}

df_investor['transaction_type'] = (
    df_investor['transaction_type']
    .str.strip()
    .str.lower()
    .replace(transaction_map)
)

# Keep only valid amounts (> 0)
df_investor = df_investor[df_investor['amount_inr'] > 0]

# Clean KYC status
df_investor['kyc_status'] = (
    df_investor['kyc_status']
    .str.strip()
    .str.title()
)

# Convert transaction_date to datetime
df_investor['transaction_date'] = pd.to_datetime(
    df_investor['transaction_date'],
    errors='coerce'
)

# Remove rows with invalid dates
df_investor = df_investor.dropna(subset=['transaction_date'])

# Final validation checks
print("Transaction Types:")
print(df_investor['transaction_type'].unique())

print("\nKYC Status Values:")
print(df_investor['kyc_status'].unique())



print("\nTransaction Date Data Type:")
print(df_investor['transaction_date'].dtype)

# Save cleaned file
df_investor.to_csv(
    "../data/processed/clean_investor_transactions.csv",
    index=False
)

print("\nCleaned file saved successfully!")

Transaction Types:
<StringArray>
['SIP', 'Redemption', 'Lumpsum']
Length: 3, dtype: str

KYC Status Values:
<StringArray>
['Verified', 'Pending']
Length: 2, dtype: str

Transaction Date Data Type:
datetime64[us]

Cleaned file saved successfully!


In [38]:
import pandas as pd

# Read file
df_scheme = pd.read_csv("../data/raw/07_scheme_performance.csv")

In [29]:

# Convert return columns to numeric
return_cols = ["return_1yr_pct", "return_3yr_pct", "return_5yr_pct"]

for col in return_cols:
    df_scheme[col] = pd.to_numeric(df_scheme[col], errors="coerce")

# Find rows with invalid return values
invalid_returns = df_scheme[df_scheme[return_cols].isnull().any(axis=1)]

print("Rows with invalid return values:")
print(invalid_returns)

# Flag negative Sharpe ratios
df_scheme["negative_sharpe_flag"] = df_scheme["sharpe_ratio"] < 0

print("\nNegative Sharpe Ratio Records:")
print(df_scheme[df_scheme["negative_sharpe_flag"]])

# Validate expense ratio range (0.1% to 2.5%)
df_scheme["expense_ratio_flag"] = (
    (df_scheme["expense_ratio_pct"] < 0.1)
    | (df_scheme["expense_ratio_pct"] > 2.5)
)

print("\nExpense Ratio Out of Range:")
print(df_scheme[df_scheme["expense_ratio_flag"]])

# Summary
print("\nTotal Invalid Return Records:", len(invalid_returns))
print("Total Negative Sharpe Ratios:", df_scheme["negative_sharpe_flag"].sum())
print("Total Expense Ratio Violations:", df_scheme["expense_ratio_flag"].sum())

# Save cleaned/validated file
df_scheme.to_csv(
    "../data/processed/clean_scheme_performance.csv",
    index=False
)

print("\nValidation completed successfully!")

Rows with invalid return values:
Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade]
Index: []

Negative Sharpe Ratio Records:
Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade, negative_sharpe_flag]
Index: []

Expense Ratio Out of Range:
Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade, negative_sha

In [34]:
import pandas as pd

invest_df = pd.read_csv("../data/processed/clean_investor_transactions.csv")
print("clean_investor_transactions.csv:", invest_df.columns.tolist())

scheme_df = pd.read_csv("../data/processed/clean_scheme_performance.csv")
print("\nclean_scheme_performance.csv:", scheme_df.columns.tolist())

nav_df = pd.read_csv("../data/processed/nav_history_clean.csv")
print("\nnav_history_clean.csv:", nav_df.columns.tolist())

aum_df = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")
print("\n03_aum_by_fund_house.csv:", aum_df.columns.tolist())

fund_df = pd.read_csv("../data/raw/01_fund_master.csv")
print("\n01_fund_master.csv:", fund_df.columns.tolist())

clean_investor_transactions.csv: ['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

clean_scheme_performance.csv: ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade', 'negative_sharpe_flag', 'expense_ratio_flag']

nav_history_clean.csv: ['amfi_code', 'date', 'nav']

03_aum_by_fund_house.csv: ['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']

01_fund_master.csv: ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_co

In [35]:
import os

path = "../data/raw"

files = os.listdir(path)

print("Files in directory:")
for f in files:
    print(f)

Files in directory:
01_fund_master.csv
02_nav_history.csv
03_aum_by_fund_house.csv
04_monthly_sip_inflows.csv
05_category_inflows.csv
06_industry_folio_count.csv
07_scheme_performance.csv
08_investor_transactions.csv
09_portfolio_holdings.csv
10_benchmark_indices.csv
axis_bluechip.csv
hdfc_top100_nav.csv
icici_bluechip.csv
kotak_bluechip.csv
nippon_largecap.csv
sbi_bluechip.csv


In [36]:
import os

path = "../data/processed"

files = os.listdir(path)

print("Files in directory:")
for f in files:
    print(f)

Files in directory:
clean_investor_transactions.csv
clean_scheme_performance.csv
nav_history_clean.csv
